# 05 — Full model with profiled qqZZ normalizations

Use the [NSBI LHC toolkit](https://github.com/iris-hep/nsbi-lhc-toolkit/) for the unbinned extended likelihood, fits, profile scans, and likelihood curves. We fit **one POI, $\mu$, and three free nuisance parameters** $\mu_{\mathrm{qqZZ0}},\mu_{\mathrm{qqZZ1}},\mu_{\mathrm{qqZZ2}}$. There are **no nuisance constraint terms** or other systematic/MC-statistical uncertainties.

With $d=-10+\sqrt{10}$, the physical coefficients are:

| Process | Coefficient |
|---|---|
| SBI | $\sqrt\mu$ |
| S | $\mu-\sqrt\mu$ |
| B | $1-\sqrt\mu$ |
| EWB | $[(1-\sqrt{10})\mu+9\sqrt\mu-10+\sqrt{10}]/d$ |
| EWSBI | $[\sqrt{10}\mu-10\sqrt\mu]/d$ |
| EWSBI10 | $[-\mu+\sqrt\mu]/d$ |
| qqZZ, 0 jets | $\mu_{\mathrm{qqZZ0}}$ |
| qqZZ, 1 jet | $\mu_{\mathrm{qqZZ1}}$ |
| qqZZ, ≥2 jets | $\mu_{\mathrm{qqZZ2}}$ |
| ttV | $1$ |

The reference remains **S + EWSBI10**. The SM Asimov is **SBI + EWSBI + qqZZ + ttV**, at $(\mu,\mu_{\mathrm{qqZZ0}},\mu_{\mathrm{qqZZ1}},\mu_{\mathrm{qqZZ2}})=(1,1,1,1)$.

Repeat notebook 04's two comparisons:

1. Full-reference reweighted Asimov versus **2,048; 16,384; 131,072; 1,048,576** nested random reference events, recalculating all Z factors for each size.
2. Full-reference reweighted Asimov versus a **direct SBI + EWSBI + qqZZ + ttV Asimov**, using native event weights and the same full-reference model normalization.

All scans cover **$0\leq\mu\leq1.8$**, in steps of 0.025. Full native process yields stay fixed when reducing the reference count. All three qqZZ normalizations are **profiled at every scan point**.

## Setup

Use a CPU runtime, as in 04. If the version check reports a stale import, choose **Runtime → Restart session**, then **Run all**. The toolkit is pinned to the same inspected commit.

The NumPy selections and normalization are below. `utils_full_likelihood.py` formats the workspace, checks its physical domain, and maps four physical fit parameters to the toolkit's normfactors. `utils_likelihood.py` supplies the existing ATLAS plot formatting. The likelihood and minimization are calculated by the toolkit.

In [ ]:
# This notebook uses CPU JAX. Remove only installed CUDA JAX plugin packages.
import sys
import subprocess
from importlib.metadata import distributions
cuda_names = {"jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"}
installed_cuda = [dist.metadata["Name"] for dist in distributions()
                  if dist.metadata.get("Name", "").lower() in cuda_names]
if installed_cuda:
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *installed_cuda])

%pip -q install "matplotlib==3.10.0" "mplhep==0.4.1" tables "jax[cpu]==0.6.2" "iminuit==2.33.0" "scikit-learn==1.9.1" "git+https://github.com/iris-hep/nsbi-lhc-toolkit.git@fc09848fc6540fd32310faebbe9db6eea7ecd17b"


In [ ]:
import sys
import os
from importlib.metadata import version

packages = {name: name for name in (
    "numpy", "pandas", "matplotlib", "mplhep", "tables", "jax", "jaxlib", "iminuit", "scipy"
)}
packages["sklearn"] = "scikit-learn"
for module_name, package in packages.items():
    loaded = sys.modules.get(module_name)
    if loaded is not None and loaded.__version__ != version(package):
        raise RuntimeError(
            f"{package} changed on disk while it was loaded. "
            "Choose Runtime > Restart session, then Run all."
        )
os.environ["JAX_PLATFORMS"] = "cpu"

import gc
import json
import importlib
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
import jax
from IPython.display import display
from nsbi_common_utils import models, inference

try:
    from google.colab import drive
except ImportError:
    import utils as offshell_utils
    import utils_likelihood
    import utils_full_likelihood
else:
    drive.mount("/content/drive")
    repo_url = "https://raw.githubusercontent.com/rafaellopesdesa/offshell_reweighting/main"
    urlretrieve(f"{repo_url}/utils.py", "/content/offshell_utils.py")
    urlretrieve(f"{repo_url}/utils_likelihood.py", "/content/utils_likelihood.py")
    urlretrieve(f"{repo_url}/utils_full_likelihood.py", "/content/utils_full_likelihood.py")
    import offshell_utils
    import utils_likelihood
    import utils_full_likelihood

importlib.invalidate_caches()
importlib.reload(offshell_utils)
importlib.reload(utils_likelihood)
importlib.reload(utils_full_likelihood)
hep.style.use("ATLAS")
offshell_utils.check_plot_exports()
print("Toolkit commit:", utils_likelihood.TOOLKIT_COMMIT)
print("JAX devices:", jax.devices())
print("PNG and PDF export checks passed.")


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/HZZ_OffShell")
DATA_PATH = PROJECT_DIR / "dataset_wDR_mean_origIndex.h5"
RESULT_DIR = PROJECT_DIR / "likelihoods" / "05_full_model"
PLOT_DIR = PROJECT_DIR / "plots" / "05_full_model"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSES = ["SBI", "S", "B", "EWB", "EWSBI", "EWSBI10", "qqZZ", "ttV"]
ASIMOV_PROCESSES = ["SBI", "EWSBI", "qqZZ", "ttV"]
PARAMETERS = ["mu", "mu_qqZZ0", "mu_qqZZ1", "mu_qqZZ2"]
SUBSET_SIZES = [2048, 16384, 131072, 1048576]
RANDOM_SEED = 42
SCAN_RANGE = (0.0, 1.8)
SCAN_POINTS = 73  # Step 0.025, including mu=1 exactly.


## Load the samples and fix the physical yields

Retain the finite-value selection from 01–04: finite `weight_kFact`, `NN_disc_New`, and `m4l_fsr`, with no histogram-range cuts. Read all eight stored ratio branches on reference and direct Asimov events. Signed event weights are retained.

The qqZZ categories depend on **the jets of each event where the likelihood is evaluated**, including reference events. Check that `n_jets` is a nonnegative integer before forming the 0, 1, and ≥2 categories. Invalid jet counts or nonfinite ratios stop the notebook instead of silently changing the sample.

**Ntuple labeling:** select ttV with `subType == "ttV"`; select every other process with `Type == process_name`. The same ttV mask is used for its full native yield and for the direct Asimov sample. The stored density-ratio branch remains `r_ttV`.

In [ ]:
data = offshell_utils.load_columns(DATA_PATH, [
    "Type", "subType", "weight_kFact", "NN_disc_New", "m4l_fsr", "n_jets",
    *[f"r_{name}" for name in PROCESSES],
])
event_type = data["Type"]
is_ttv = data["subType"] == "ttV"  # This process is labeled by subType in the ntuple.
weights = np.asarray(data["weight_kFact"], dtype=np.float64)
finite = (np.isfinite(weights) & np.isfinite(data["NN_disc_New"])
          & np.isfinite(data["m4l_fsr"]))
reference = np.isin(event_type, ["S", "EWSBI10"]) & finite
direct_types = [name for name in ASIMOV_PROCESSES if name != "ttV"]
direct = (np.isin(event_type, direct_types) | is_ttv) & finite
used = reference | direct
jets = np.asarray(data["n_jets"])
if not np.all(np.isfinite(jets[used]) & (jets[used] >= 0) & (jets[used] == np.floor(jets[used]))):
    raise ValueError("Expected finite, nonnegative integer n_jets on every likelihood event.")

reference_rows = np.flatnonzero(reference)
w_ref = weights[reference]
jet_ref = np.minimum(jets[reference], 2).astype(np.int8)
r_ref = {name: np.asarray(data[f"r_{name}"][reference], dtype=np.float64) for name in PROCESSES}
direct_rows = np.flatnonzero(direct)
w_direct = weights[direct]
jet_direct = np.minimum(jets[direct], 2).astype(np.int8)
r_direct = {name: np.asarray(data[f"r_{name}"][direct], dtype=np.float64) for name in PROCESSES}
for sample_name, ratio_map in [("reference", r_ref), ("direct", r_direct)]:
    for name, ratios in ratio_map.items():
        if not np.isfinite(ratios).all():
            raise ValueError(f"Nonfinite r_{name} on {sample_name} events; inspect the input.")

native_yields = {}
yield_rows = []
for name in ["Reference"] + PROCESSES:
    if name == "Reference":
        original = np.isin(event_type, ["S", "EWSBI10"])
    elif name == "ttV":
        original = is_ttv
    else:
        original = event_type == name
    mask = original & finite
    total = float(weights[mask].sum())
    native_yields[name] = total
    yield_rows.append({"sample": name, "events": int(mask.sum()), "yield": total,
                       "excluded_nonfinite": int(np.count_nonzero(original & ~finite)),
                       "negative_weights": int(np.count_nonzero(weights[mask] < 0))})
yield_table = pd.DataFrame(yield_rows).set_index("sample")
display(yield_table)
yield_table.to_csv(RESULT_DIR / "native_yields.csv")
if any(not np.isfinite(value) or value <= 0 for value in native_yields.values()):
    raise ValueError("Each physical sample needs a finite, positive total signed yield.")
if max(SUBSET_SIZES) > len(w_ref):
    raise ValueError(f"Only {len(w_ref):,} reference events; cannot draw all requested sizes.")
expected_yield = sum(native_yields[name] for name in ASIMOV_PROCESSES)
print(f"SM Asimov yield: {expected_yield:.6f}")
del data, event_type, is_ttv, weights, finite, reference, direct, used, jets, original, mask, ratios, ratio_map
gc.collect()


## Same-sample normalization and the qqZZ jet split

For each reference size, calculate
$$
u_i=\frac{w_i}{\sum_{\rm ref}w},\qquad Z_j=\sum_{\rm ref}u_i r_j(x_i),\qquad
\widehat r_j=r_j/Z_j.
$$
Each of the **eight** corrected ratios has weighted mean one. At the SM point, the Asimov weights are
$$a_i=u_i\left(\lambda_{\rm SBI}\widehat r_{\rm SBI}
+\lambda_{\rm EWSBI}\widehat r_{\rm EWSBI}
+\lambda_{\rm qqZZ}\widehat r_{\rm qqZZ}
+\lambda_{\rm ttV}\widehat r_{\rm ttV}\right)_i.$$

To implement your shared inclusive $\lambda_{\rm qqZZ}$ and $\widehat r_{\rm qqZZ}$ in three workspace processes, let $I_c$ select jet category $c$ and calculate
$$f_c=\sum_{\rm ref}u_i\widehat r_{\rm qqZZ}(x_i)I_c(x_i),\qquad
\lambda_{\rm qqZZ,c}=\lambda_{\rm qqZZ}f_c,\qquad
\widehat r_{\rm qqZZ,c}=\widehat r_{\rm qqZZ}I_c/f_c.$$
Then $\lambda_{\rm qqZZ,c}\widehat r_{\rm qqZZ,c}=\lambda_{\rm qqZZ}\widehat r_{\rm qqZZ}I_c$, exactly your event-level prescription. Each category ratio has mean one and $\sum_c f_c=1$. The profiled qqZZ rate is $\lambda_{\rm qqZZ}\sum_c\mu_{\rm qqZZ,c}f_c$.

Recalculate both **Z factors and jet fractions** for every reference subset. The inclusive native qqZZ yield stays fixed; its category fractions are reference-based integration estimates. Once a workspace is built, its fractions stay fixed during profiling. This also avoids counting the inclusive qqZZ yield three times.

In [ ]:
permutation = np.random.default_rng(RANDOM_SEED).permutation(len(w_ref))
reference_cases = [("full", len(w_ref))] + [(f"n_{n}", n) for n in SUBSET_SIZES]
workspace_paths = {}
normalization_rows, jet_rows = [], []

for label, n in reference_cases:
    selected = np.arange(n) if label == "full" else permutation[:n]
    sample_dir = RESULT_DIR / label
    sample_dir.mkdir(parents=True, exist_ok=True)
    selected_weight_sum = float(w_ref[selected].sum())
    if not np.isfinite(selected_weight_sum) or selected_weight_sum == 0:
        raise ValueError(f"{label}: reference weights cannot be normalized.")
    u = w_ref[selected] / selected_weight_sum
    np.testing.assert_allclose(u.sum(), 1.0, rtol=1e-10)

    z, corrected = {}, {}
    for name in PROCESSES:
        r = r_ref[name][selected]
        z[name] = float(np.sum(u * r))
        if not np.isfinite(z[name]) or z[name] <= 0:
            raise ValueError(f"{label}: invalid Z_{name} = {z[name]}")
        corrected[name] = r / z[name]
        mean_after = float(np.sum(u * corrected[name]))
        np.testing.assert_allclose(mean_after, 1.0, rtol=1e-10)
        np.save(sample_dir / f"r_{name}.npy", corrected[name])
        normalization_rows.append({"sample": label, "events": n, "process": name,
                                   "Z": z[name], "corrected_mean": mean_after})

    fractions = []
    for c in range(3):
        in_category = jet_ref[selected] == c
        masked_ratio = corrected["qqZZ"] * in_category
        fraction = float(np.sum(u * masked_ratio))
        if not np.isfinite(fraction) or fraction <= 0:
            raise ValueError(f"{label}: qqZZ category {c} has a nonpositive reference integral.")
        category_ratio = masked_ratio / fraction
        np.testing.assert_allclose(np.sum(u * category_ratio), 1.0, rtol=1e-10)
        np.save(sample_dir / f"r_qqZZ{c}.npy", category_ratio)
        fractions.append(fraction)
        jet_rows.append({"sample": label, "category": c, "fraction": fraction,
                         "yield": native_yields["qqZZ"] * fraction,
                         "corrected_mean": float(np.sum(u * category_ratio))})
    np.testing.assert_allclose(sum(fractions), 1.0, rtol=1e-10)
    if label == "full":
        full_z, full_fractions = z.copy(), fractions.copy()

    intensity_sm = sum(native_yields[name] * corrected[name] for name in ASIMOV_PROCESSES)
    if not np.all(intensity_sm > 0):
        raise ValueError(f"{label}: nonpositive full-model intensity at the SM point.")
    asimov = u * intensity_sm
    np.testing.assert_allclose(asimov.sum(), expected_yield, rtol=1e-10)
    np.save(sample_dir / "asimov_weights.npy", asimov)
    np.save(sample_dir / "reference_weights_normalized.npy", u)
    np.save(sample_dir / "event_rows.npy", reference_rows[selected])
    np.save(sample_dir / "jet_category.npy", jet_ref[selected])
    workspace_paths[label] = utils_full_likelihood.write_workspace(
        sample_dir, native_yields, fractions,
        metadata={"reference_events": n, "reference_types": ["S", "EWSBI10"],
                  "asimov_source": "reweighted_reference", "asimov_events": n,
                  "asimov_processes": ASIMOV_PROCESSES,
                  "ttV_selection": "subType == 'ttV'", "normalization_reference": label,
                  "scan_range": list(SCAN_RANGE), "random_seed": RANDOM_SEED, "Z": z,
                  "selected_reference_weight_sum": selected_weight_sum,
                  "asimov_yield": float(asimov.sum()),
                  "negative_asimov_weights": int(np.count_nonzero(asimov < 0)),
                  "input_file": str(DATA_PATH.resolve())},
    )
    print(f"{label}: {n:,} reference events; Asimov yield = {asimov.sum():.6f}")

normalization_table = pd.DataFrame(normalization_rows)
jet_table = pd.DataFrame(jet_rows)
display(normalization_table.set_index(["sample", "process"]))
display(jet_table.set_index(["sample", "category"]))
normalization_table.to_csv(RESULT_DIR / "normalization_checks.csv", index=False)
jet_table.to_csv(RESULT_DIR / "qqZZ_jet_fractions.csv", index=False)
del r_ref, jet_ref, reference_rows, corrected, r, asimov, intensity_sm, u, selected
del masked_ratio, category_ratio, in_category, permutation
gc.collect()


## Direct SBI + EWSBI + qqZZ + ttV Asimov

Use the native events with their original `weight_kFact`. Evaluate each ratio on these events, dividing by its **full-reference Z**. Apply the jet masks on these direct events, while retaining the **full-reference jet fractions** in the model. Neither Z nor the fractions are recalibrated on the direct Asimov sample.

The two full-reference-normalized models are identical; only the Asimov integration events and weights change. Their total Asimov yields agree. The direct fit's minimum and nuisance estimates can move away from one, revealing finite-MC or ratio-model closure differences.

In [ ]:
direct_dir = RESULT_DIR / "direct"
direct_dir.mkdir(parents=True, exist_ok=True)
for name in PROCESSES:
    np.save(direct_dir / f"r_{name}.npy", r_direct[name] / full_z[name])
for c in range(3):
    category_ratio = (r_direct["qqZZ"] / full_z["qqZZ"]) * (jet_direct == c) / full_fractions[c]
    np.save(direct_dir / f"r_qqZZ{c}.npy", category_ratio)
np.testing.assert_allclose(w_direct.sum(), expected_yield, rtol=1e-10)
np.save(direct_dir / "asimov_weights.npy", w_direct)
np.save(direct_dir / "event_rows.npy", direct_rows)
np.save(direct_dir / "jet_category.npy", jet_direct)
workspace_paths["direct"] = utils_full_likelihood.write_workspace(
    direct_dir, native_yields, full_fractions,
    metadata={"reference_events": len(w_ref), "reference_types": ["S", "EWSBI10"],
              "asimov_source": "native_SBI_EWSBI_qqZZ_ttV", "asimov_events": len(w_direct),
              "asimov_processes": ASIMOV_PROCESSES,
              "ttV_selection": "subType == 'ttV'", "normalization_reference": "full",
              "Z": full_z, "scan_range": list(SCAN_RANGE),
              "asimov_yield": float(w_direct.sum()),
              "negative_asimov_weights": int(np.count_nonzero(w_direct < 0)),
              "input_file": str(DATA_PATH.resolve())},
)
cases = reference_cases + [("direct", len(w_direct))]
print(f"Direct full-model Asimov: {len(w_direct):,} events; yield = {w_direct.sum():.6f}")
del r_direct, w_direct, direct_rows, jet_direct, category_ratio
gc.collect()


## Fit and profile with the toolkit

The workspace has ten processes and nine internal normfactors: six tied to $\mu$, and three independent qqZZ factors. The adapter exposes **four physical parameters**, in the order shown below. ttV has no modifier. Every workspace modifier is an unconstrained `normfactor`; there are no Gaussian penalties.

The qqZZ parameters are free to vary over their physical nonnegative domain, with no upper bound. At each fixed $\mu$, the toolkit minimizes over all three of them. We do not freeze them for the stat-only scan. Negative **interference coefficients** remain allowed.

Before fitting, check the total intensity over the complete $[0,1.8]$ interval at nominal qqZZ factors, including any interior minimum of its quadratic in $\sqrt\mu$. During profiling, nonphysical parameter points return an infinite objective; no density, event weight, or coefficient is clipped. If the nominal domain check or a profile minimization fails, stop and inspect the saved diagnostics.

The toolkit returns $t_\mu=-2\Delta\log L$, subtracting each curve's **scan-grid minimum**. The fitted minimum and all three fitted qqZZ normalizations are saved separately. The direct best fit need not lie exactly on the scan grid.

In [ ]:
scan_grid = np.linspace(*SCAN_RANGE, SCAN_POINTS)
if SCAN_RANGE[0] < 0 or not np.any(np.isclose(scan_grid, 1.0, atol=1e-12, rtol=0)):
    raise ValueError("Use a nonnegative scan grid containing mu=1 exactly.")

scans, fit_rows, domain_rows = {}, [], []
for label, n in cases:
    print(f"\n--- {label}: {n:,} Asimov events ---")
    domain = utils_full_likelihood.check_workspace_domain(workspace_paths[label], SCAN_RANGE)
    domain_rows.append({"sample": label, **domain})
    print(f"Minimum nominal intensity over the full interval: {domain['min_intensity']:.6g}")
    workspace = json.loads(workspace_paths[label].read_text())
    model = models.sbi_parametric_model(workspace, measurement_to_fit="full_stat_only")
    nll = utils_full_likelihood.physical_likelihood(model, mu_bounds=SCAN_RANGE)
    if not np.isfinite(nll([1.0, 1.0, 1.0, 1.0])):
        raise ValueError(f"{label}: nonphysical total intensity at the SM point.")

    fitter = inference.inference(
        model_nll=nll, initial_values=[1.0, 1.0, 1.0, 1.0],
        list_parameters=PARAMETERS, num_unconstrained_params=4,
    )
    fitter.perform_fit()
    if not np.isfinite(nll(fitter.pulls_global_fit)):
        raise ValueError(f"{label}: nonfinite best-fit likelihood; inspect the fit output.")
    mu, t_mu = utils_full_likelihood.profile_scan(fitter, SCAN_RANGE, SCAN_POINTS)
    scans[label] = (mu, t_mu)
    pd.DataFrame({"mu": mu, "t_mu": t_mu}).to_csv(
        workspace_paths[label].parent / "scan.csv", index=False,
    )
    fit_rows.append({"sample": label, "asimov_events": n,
                     "reference_events": workspace["metadata"]["reference_events"],
                     **{f"{name}_hat": float(value) for name, value in zip(PARAMETERS, fitter.pulls_global_fit)},
                     "grid_minimum_mu": float(mu[np.argmin(t_mu)]),
                     "asimov_yield": workspace["metadata"]["asimov_yield"]})
    del fitter, nll, model
    jax.clear_caches()
    gc.collect()

domain_table = pd.DataFrame(domain_rows).set_index("sample")
display(domain_table[["min_intensity", "mu_at_min_intensity", "nonpositive_events", "min_total_yield"]])
domain_table.to_csv(RESULT_DIR / "domain_checks.csv")
fit_table = pd.DataFrame(fit_rows).set_index("sample")
display(fit_table)
fit_table.to_csv(RESULT_DIR / "fit_summary.csv")


## Compare reference sizes and Asimov constructions

Each curve profiles the same three qqZZ normalizations. The first figure tests reference-size convergence with per-sample normalization. The second tests full-reference ratio closure against the native full-model Asimov. Both use the toolkit's likelihood-curve plotting with ATLAS styling.

In [ ]:
plot_groups = [
    ("reference_size_comparison", [label for label, _ in reference_cases],
     [f"Full reference ({len(w_ref):,})"] + [f"{n:,} reference events" for n in SUBSET_SIZES]),
    ("asimov_comparison", ["full", "direct"],
     ["Full reference reweighted", "Direct SBI + EWSBI + qqZZ + ttV"]),
]
for filename, keys, labels in plot_groups:
    fig, ax = utils_likelihood.plot_likelihood_scans(
        [scans[key] for key in keys], labels, SCAN_RANGE,
    )
    ax.text(0.98, 1.06, "3 free qqZZ normalizations profiled",
            transform=ax.transAxes, ha="right", fontsize=10)
    for extension in ("png", "pdf"):
        fig.savefig(PLOT_DIR / f"{filename}.{extension}", dpi=180, bbox_inches="tight")
    plt.show()
print("Workspaces, arrays, Z factors, jet fractions, domain checks, and scans:", RESULT_DIR)
print("Likelihood figures:", PLOT_DIR)


There should be 40 inclusive normalization checks and 15 jet-category checks, all with corrected mean one. All six Asimov yields should agree. The five reference Asimov fits should recover approximately `(mu, mu_qqZZ0, mu_qqZZ1, mu_qqZZ2) = (1, 1, 1, 1)`; the direct fit is allowed to differ.

To reopen a saved model, load `workspace.json`, create `models.sbi_parametric_model(workspace, "full_stat_only")`, then wrap it with `utils_full_likelihood.physical_likelihood(model)` and fit the four parameters in `PARAMETERS` order. The adapter reads the saved scan range. Never fit the six signal/interference normfactors independently.